In [1]:
import pandas as pd
import numpy as np

# ============================
# 설정
# ============================

FEVD_PATH = "./fevd_final_horizon.csv"

OUTPUT_SPILLOVER_MATRIX = "./spillover_matrix.csv"
OUTPUT_SPILLOVER_INDEX = "./spillover_index.csv"
OUTPUT_PAIRWISE_NET = "./pairwise_net_spillover.csv"

# ============================
# FEVD 불러오기
# ============================

fevd = pd.read_csv(FEVD_PATH, index_col=0)

# 숫자형 변환
fevd = fevd.apply(pd.to_numeric, errors="coerce")

variables = fevd.index.tolist()
n = len(variables)

print("Loaded FEVD matrix:")
print(fevd.round(4))

# ============================
# Spillover matrix
# - diagonal 제거한 off-diagonal matrix
# ============================

spill = fevd.copy()
np.fill_diagonal(spill.values, 0.0)

spill.to_csv(OUTPUT_SPILLOVER_MATRIX)
print(f"\nSaved spillover matrix: {OUTPUT_SPILLOVER_MATRIX}")

# ============================
# Total Spillover Index
# Diebold-Yilmaz:
# off-diagonal sum / total sum * 100
# ============================

off_diag_sum = spill.values.sum()
total_sum = fevd.values.sum()

total_spillover_index = (off_diag_sum / total_sum) * 100

# ============================
# Directional Spillover
# FROM_i = row sum of off-diagonal
# TO_i   = column sum of off-diagonal
# NET_i  = TO_i - FROM_i
# ============================

from_others = spill.sum(axis=1) * 100
to_others = spill.sum(axis=0) * 100
net_spillover = to_others - from_others

own_share = np.diag(fevd.values) * 100

spillover_index_df = pd.DataFrame({
    "OWN_%": own_share,
    "FROM_%": from_others,
    "TO_%": to_others,
    "NET_%": net_spillover
}, index=variables)

# total spillover는 별도 row로 추가
total_row = pd.DataFrame({
    "OWN_%": [np.nan],
    "FROM_%": [np.nan],
    "TO_%": [np.nan],
    "NET_%": [total_spillover_index]
}, index=["TOTAL_SPILLOVER_INDEX_%"])

spillover_index_df = pd.concat([spillover_index_df, total_row])

spillover_index_df.to_csv(OUTPUT_SPILLOVER_INDEX)
print(f"Saved spillover summary: {OUTPUT_SPILLOVER_INDEX}")

# ============================
# Pairwise Net Spillover
# pairwise(i,j) = theta_ji - theta_ij
# 해석:
# 양수이면 i -> j 방향 전이가 더 큼
# ============================

pairwise_net = pd.DataFrame(
    np.zeros((n, n)),
    index=variables,
    columns=variables
)

for i in range(n):
    for j in range(n):
        if i == j:
            pairwise_net.iloc[i, j] = 0.0
        else:
            # i -> j  : row=j, col=i
            # j -> i  : row=i, col=j
            i_to_j = spill.iloc[j, i]
            j_to_i = spill.iloc[i, j]
            pairwise_net.iloc[i, j] = (i_to_j - j_to_i) * 100

pairwise_net.to_csv(OUTPUT_PAIRWISE_NET)
print(f"Saved pairwise net spillover: {OUTPUT_PAIRWISE_NET}")

# ============================
# 주요 결과 출력
# ============================

print("\n===== Diebold-Yilmaz Spillover Summary =====")
print(f"Total Spillover Index: {total_spillover_index:.4f}%\n")

print("Directional Spillover:")
print(spillover_index_df.round(4))

print("\nPairwise Net Spillover (%):")
print(pairwise_net.round(4))

# ============================
# SOLVPN ↔ COPPER 관심 관계 출력
# ============================

if "dlog_SOLVPN" in variables and "dlog_COPPER" in variables:
    solvpn_to_copper = spill.loc["dlog_COPPER", "dlog_SOLVPN"] * 100
    copper_to_solvpn = spill.loc["dlog_SOLVPN", "dlog_COPPER"] * 100
    net_pair = solvpn_to_copper - copper_to_solvpn

    print("\n===== SOLVPN ↔ COPPER =====")
    print(f"SOLVPN -> COPPER : {solvpn_to_copper:.4f}%")
    print(f"COPPER -> SOLVPN : {copper_to_solvpn:.4f}%")
    print(f"Pairwise Net (SOLVPN -> COPPER) : {net_pair:.4f}%")

Loaded FEVD matrix:
             dlog_SOLVPN  dlog_COPPER  dlog_DXY  d_UST10Y  dlog_VIX
dlog_SOLVPN       0.9943       0.0000    0.0002    0.0028    0.0027
dlog_COPPER       0.0537       0.9377    0.0014    0.0001    0.0072
dlog_DXY          0.0978       0.0703    0.8213    0.0069    0.0036
d_UST10Y          0.0529       0.0031    0.0868    0.8568    0.0005
dlog_VIX          0.2832       0.0152    0.0030    0.0116    0.6870

Saved spillover matrix: ./spillover_matrix.csv
Saved spillover summary: ./spillover_index.csv
Saved pairwise net spillover: ./pairwise_net_spillover.csv

===== Diebold-Yilmaz Spillover Summary =====
Total Spillover Index: 14.0578%

Directional Spillover:
                           OWN_%   FROM_%     TO_%    NET_%
dlog_SOLVPN              99.4333   0.5667  48.7585  48.1918
dlog_COPPER              93.7693   6.2307   8.8656   2.6349
dlog_DXY                 82.1295  17.8705   9.1291  -8.7415
d_UST10Y                 85.6811  14.3189   2.1377 -12.1812
dlog_VIX        